# FTPrimitiveBench — primitives & noise models

End-to-end walkthrough of the public API. Two halves:

1. Build a noiseless `stim.Circuit` for each primitive (`memory`, `transversal_h`, `lattice_surgery`, `s_gate`).
2. Configure a noise model and apply it: `noisy = model.noisy_circuit(clean)`.

Visualization helpers (matplotlib 2D + plotly 3D) live in a separate code-only notebook at [`tutorials/visualization_tutorial.ipynb`](visualization_tutorial.ipynb).

To run this notebook from a fresh checkout:

```bash
pip install -e ".[viz,dev]"
jupyter notebook tutorials/noise_model_tutorial.ipynb
```

## Setup

In [ ]:
from ft_primitive_bench.surface_code.circuits import (
    memory,
    transversal_h,
    lattice_surgery,
    s_gate,
)
from ft_primitive_bench.noise_models import (
    Coherence,
    NoiseProfile,
    noise_model,
    uniform_depolarizing,
    pauli_biased,
    measurement_biased,
    nonuniform,
    strip_noise_channels,
)

## 1. Noiseless primitive circuits

Each call returns a `stim.Circuit` with `DETECTOR` and `OBSERVABLE_INCLUDE` annotations already in place.

### 1.1 `memory`

`N` rounds of stabilizer measurement on a single patch, then a basis-aligned destructive readout.

- `x_distance`, `z_distance` — patch dimensions (X / Z code distance).
- `rounds` — number of stabilizer rounds before the final readout.
- `meas_basis` — `"Z"` or `"X"`; sets both initialization and final-readout basis.

In [ ]:
memory_circ = memory(x_distance=3, z_distance=3, rounds=3, meas_basis="Z")


### 1.2 `transversal_h`

`pre_rounds` of stabilizer measurement, one transversal Hadamard layer (with automatic X↔Z schedule swap), then `post_rounds` more. Total cycles = `pre_rounds + 1 + post_rounds`.

- `x_distance`, `z_distance` — patch dimensions.
- `pre_rounds` — rounds before the H layer.
- `post_rounds` — rounds after the H layer.
- `meas_basis` — initialization / readout basis (the H swaps stabilizer roles internally).

In [ ]:
H_circ = transversal_h(
    x_distance=3, z_distance=3,
    pre_rounds=2, post_rounds=2, meas_basis="Z",
)


### 1.3 `lattice_surgery`

Two patches separated by a configurable bridge, with MZZ / MXX merge and split phases.

- `x_distance`, `z_distance` — per-patch dimensions.
- `bridge_length` — data-row gap between the two patches.
- `pre_rounds` — rounds with the two patches independent.
- `merge_rounds` — rounds with the merged stabilizer active.
- `post_rounds` — rounds after split, patches independent again.
- `meas_basis` — `"Z"` (MZZ, vertical stack) or `"X"` (MXX, horizontal stack).

#### Parity rule

`lattice_surgery(...)` requires the bridge length plus the patch dimension *along the merging axis* to be **even**:

- `meas_basis="Z"` (MZZ, vertical stack) → `(z_distance + bridge_length) % 2 == 0`
- `meas_basis="X"` (MXX, horizontal stack) → `(x_distance + bridge_length) % 2 == 0`

This keeps the merged-patch checkerboard parity aligned with the two individual patches. Calling `lattice_surgery(...)` (or `s_gate(...)`, which uses lattice surgery internally and inherits the same constraint) with an odd sum raises `ValueError`. If you have a target bridge length, bump the patch dimension along the merging axis (or vice versa) to satisfy the rule.

```python
# OK: z_distance + bridge_length = 3 + 1 = 4, even
lattice_surgery(x_distance=3, z_distance=3, bridge_length=1,
                pre_rounds=1, merge_rounds=2, post_rounds=1, meas_basis="Z")

# ValueError: 3 + 2 = 5 is odd
lattice_surgery(x_distance=3, z_distance=3, bridge_length=2,
                pre_rounds=1, merge_rounds=2, post_rounds=1, meas_basis="Z")
```

In [ ]:
lattice_surgery_circ = lattice_surgery(
    x_distance=3, z_distance=3, bridge_length=1,
    pre_rounds=2, merge_rounds=3, post_rounds=2, meas_basis="Z",
)


### 1.4 `s_gate`

ZZ lattice surgery spliced with a teleported logical-Y magic-state measurement. Inherits the `lattice_surgery` parity rule.

- `distance` — square patch distance (`x_distance == z_distance`).
- `bridge_length` — gap between the data patch and the ancilla patch.
- `pre_rounds` — rounds before the ZZ merge.
- `merge_rounds` — ZZ surgery merge phase.
- `boundary_rounds` — ancilla-patch boundary rounds during the magic-Y measurement.
- `post_rounds` — rounds after the magic measurement.

In [ ]:
S_circ = s_gate(
    distance=3, bridge_length=1,
    pre_rounds=1, merge_rounds=3,
    boundary_rounds=2, post_rounds=1,
)


## 2. Noise models

Pre-packaged factories cover the four common patterns. The canonical `noise_model(...)` factory underpins all of them and accepts per-component / per-round overrides via `NoiseProfile`. Apply with `model.noisy_circuit(clean)`.

### Idling — two modes

Idle qubits during a gate / measurement moment receive a noise channel determined by one of two modes:

- **Mode A (constant rate):** pass `p_idle` and (optionally) `p_idle_meas`. Each idle qubit gets `DEPOLARIZE1(p_idle)` during gate moments and `DEPOLARIZE1(p_idle_meas)` during measurement / reset moments.
- **Mode B (PTA from T1/T2):** pass `coherence=(T1, T2)` and bundle every per-class rate as `(rate, duration)`. The engine computes per-moment idle from the residual idle window via the Pauli-twirled approximation (`PAULI_CHANNEL_1`).

Mode A and Mode B are exclusive globally. A per-qubit `NoiseProfile` override can put one qubit in Mode B (set its `T1`/`T2` and `(rate, duration)` tuples) while the rest of the device stays in Mode A.

### 2.1 `uniform_depolarizing` — symmetric depolarizing baseline

In [ ]:
model = uniform_depolarizing(p=1e-3)
noisy = model.noisy_circuit(memory_circ)
 

### 2.2 `pauli_biased` — bias the X or Z Pauli weight

`bias_factor=1` recovers depolarizing. Larger values concentrate weight on the chosen axis.

In [ ]:
model = pauli_biased(p=1e-3, axis="Z", bias_factor=10)
noisy = model.noisy_circuit(memory_circ)
 

### 2.3 `measurement_biased` — uniform SPAM amplification

In [ ]:
model = measurement_biased(p=1e-3, bias_factor=10)
noisy = model.noisy_circuit(memory_circ)
 

### 2.4 `nonuniform` — Gaussian per-component scatter

`sigma` is the standard deviation of the multiplicative perturbation `(1 + δ)`, with `δ ~ N(0, σ²)`. `variant="space_only"` freezes one factor per qubit/pair across rounds; `"space_time"` resamples per round. Set `seed` for reproducibility.

In [ ]:
model = nonuniform(p=1e-3, sigma=0.3, variant="space_time", seed=42)
noisy = model.noisy_circuit(memory_circ)
 

### 2.5 `noise_model` — baseline + per-class rates

Each unset `p_*` defaults to `p`. Example: SI1000-style asymmetric rates.

In [ ]:
model = noise_model(
    p=1e-3,
    p_1q=1e-4, p_2q=1e-3, p_meas=5e-3, p_reset=2e-3,
    p_idle=1e-4, p_idle_meas=2e-3,
)
noisy = model.noisy_circuit(memory_circ)
 

### 2.6 `noise_model` — Mode B: T1/T2 with bundled durations

Pass `coherence=(T1, T2)` (or a `Coherence` object). Per-class rates become `(rate, duration)` tuples; the engine derives idle channels via the Pauli-twirled approximation from coherence + duration.

In [ ]:
model = noise_model(
    p=1e-3,
    p_1q=(1e-4, 40e-9), p_2q=(1e-3, 120e-9),
    p_meas=(5e-3, 200e-9), p_reset=(2e-3, 100e-9),
    coherence=Coherence(T1=30e-6, T2=20e-6),
)
noisy = model.noisy_circuit(memory_circ)
 

### 2.7 `NoiseProfile` — per-component / per-round overrides

Resolution at `(qubit q, round r)` merges:

    (None, None) → (q, None) → (None, r) → (q, r)

Higher-precedence tiers override lower-precedence tiers field-by-field. Pair entries `((q1, q2), r)` carry pair-local fields like `p_2q`.

In [ ]:
profile = NoiseProfile({
    (5, None):      {"p_meas": 1e-2},  # qubit 5, all rounds
    ((3, 4), None): {"p_2q": 5e-3},    # pair (3,4), all rounds
    (None, 1):      {"p_1q": 2e-3},    # round 1, all components
    (5, 1):         {"p_meas": 5e-2},  # qubit 5 in round 1 (highest precedence)
})
model = noise_model(p=1e-3, profile=profile)
noisy = model.noisy_circuit(memory_circ)
 

### 2.8 `noise_model` — port hardware calibration data

Drop the baseline kwargs entirely; let `profile` carry the entire spec. This is the natural shape when ingesting hardware calibration: every per-qubit / per-pair rate is in your data, no synthetic defaults.

The example below mimics an IBM/Google-style calibration export — a dict-of-dicts keyed by qubit and coupler — and assembles a `NoiseProfile` directly from it.

In [ ]:
# Synthetic per-qubit calibration export.
calibration = {
    0: {"T1": 30e-6, "T2": 22e-6, "p_meas": 4.0e-3, "p_reset": 1.0e-3, "p_1q": 1.0e-4},
    1: {"T1": 28e-6, "T2": 20e-6, "p_meas": 6.0e-3, "p_reset": 1.0e-3, "p_1q": 1.4e-4},
    5: {"T1": 25e-6, "T2": 18e-6, "p_meas": 1.5e-2, "p_reset": 2.0e-3, "p_1q": 2.0e-4},
}
couplers = {(0, 1): 2.0e-3, (3, 4): 5.0e-3}
DUR = {"p_1q": 40e-9, "p_2q": 120e-9, "p_meas": 200e-9, "p_reset": 100e-9}

profile = NoiseProfile()
for q, cal in calibration.items():
    profile[q, None] = {
        "T1": cal["T1"], "T2": cal["T2"],
        "p_1q":    (cal["p_1q"],    DUR["p_1q"]),
        "p_meas":  (cal["p_meas"],  DUR["p_meas"]),
        "p_reset": (cal["p_reset"], DUR["p_reset"]),
    }
for pair, cx in couplers.items():
    profile[pair, None] = {"p_2q": (cx, DUR["p_2q"])}

model = noise_model(profile=profile)        # no baseline — calibration is the spec
noisy = model.noisy_circuit(memory_circ)